In [ ]:
# config bootstrap (auto-added): resolve repo paths from config.py
import os as _os, sys as _sys
_h = _os.path.abspath(_os.getcwd())
while not _os.path.exists(_os.path.join(_h, 'config.py')) and _os.path.dirname(_h) != _h:
    _h = _os.path.dirname(_h)
_sys.path.insert(0, _h)
import config as _cfg

# BLIP-2 Full Pipeline — Pics Can Lie

**Sections**
1. Config & imports
2. Load dataset
3. Zero-shot evaluation (pretrained, no fine-tuning)
4. Fine-tuning
5. Post-fine-tuning evaluation
6. Compare results

## 1 — Config & Imports

In [1]:
import json
import os
import random
import time

import torch
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from torch.utils.data import DataLoader, Dataset
from transformers import AutoProcessor, Blip2ForConditionalGeneration

# ── Paths ─────────────────────────────────────────────────────────────────────
MODEL_PATH   = r"D:\Blip2\models--Salesforce--blip2-opt-2.7b\snapshots\59a1ef6c1e5117b3f65523d1c6066825bcf315e3"
ANNOTATIONS  = _os.path.join(str(_cfg.ROOT), 'datasets', 'dataset', 'data', 'NewsClipPings', 'merged_balanced', 'train.json')
METADATA     = _os.path.join(str(_cfg.ROOT), 'datasets', 'dataset', 'data', 'NewsClipPings', 'metadata', 'train.json')
IMAGES_ROOT  = _os.path.join(str(_cfg.ROOT), 'datasets', 'dataset', 'origin')
OUTPUT_DIR   = _os.path.join(str(_cfg.ROOT), 'models', 'blip2_finetuned')
EVAL_OUTPUT  = _os.path.join(str(_cfg.ROOT), 'results', 'blip2_pretrained_eval.json')

# ── Eval config ───────────────────────────────────────────────────────────────
EVAL_SAMPLES    = 500
EVAL_SEED       = 42
MAX_NEW_TOKENS  = 20

# ── Train config ──────────────────────────────────────────────────────────────
TRAIN_SAMPLES   = 5000
BATCH_SIZE      = 2
GRAD_ACCUM      = 8       # effective batch = 16
EPOCHS          = 2
LR_QFORMER      = 1e-5
LR_VISION       = 1e-6
VISION_UNFREEZE = 2
LOG_EVERY       = 50
MAX_TARGET_LEN  = 8
MAX_INPUT_LEN   = 128

os.makedirs(OUTPUT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype  = torch.float16 if torch.cuda.is_available() else torch.float32
print(f"Device: {device}  |  dtype: {dtype}")
print(f"Model path: {MODEL_PATH}")

d:\Pics Can Lie\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda  |  dtype: torch.float16
Model path: D:\Blip2\models--Salesforce--blip2-opt-2.7b\snapshots\59a1ef6c1e5117b3f65523d1c6066825bcf315e3


## 2 — Load Dataset

In [2]:
def resolve_image_path(meta_image_path: str) -> str:
    rel = meta_image_path.replace("visual_news/", "", 1)
    return os.path.join(IMAGES_ROOT, rel)


def load_and_join(annotations_path: str, metadata_path: str) -> list[dict]:
    with open(annotations_path, "r", encoding="utf-8") as f:
        annotations = json.load(f)["annotations"]
    with open(metadata_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)  # keyed by image_id (string)

    joined = []
    for ann in annotations:
        image_id = str(ann["image_id"])
        if image_id not in metadata:
            continue
        meta    = metadata[image_id]
        caption = meta.get("caption") or meta.get("title") or ""
        joined.append({
            "image_path": resolve_image_path(meta["image_path"]),
            "caption":    caption,
            "label":      int(ann["falsified"]),
        })
    return joined


all_samples = load_and_join(ANNOTATIONS, METADATA)
print(f"Total joined samples: {len(all_samples)}")

# Fixed eval split — same 500 samples before and after fine-tuning
random.seed(EVAL_SEED)
eval_samples = random.sample(all_samples, min(EVAL_SAMPLES, len(all_samples)))

# Training pool — exclude eval samples (by image_path)
eval_paths   = {s["image_path"] for s in eval_samples}
train_pool   = [s for s in all_samples if s["image_path"] not in eval_paths]
train_samples = train_pool[:TRAIN_SAMPLES]

real = sum(1 for s in eval_samples if s["label"] == 0)
ooc  = sum(1 for s in eval_samples if s["label"] == 1)
print(f"Eval  : {len(eval_samples)} samples  (real={real}, ooc={ooc})")
print(f"Train : {len(train_samples)} samples")

Total joined samples: 71072
Eval  : 500 samples  (real=250, ooc=250)
Train : 5000 samples


## 3 — Load Model & Processor

In [3]:
import glob
import json
import struct
from transformers import Blip2Config

def stream_shard_to_gpu(filepath: str, param_map: dict, device, dtype) -> int:
    """
    Read a safetensors shard with plain file I/O (no mmap) and copy each
    tensor one-at-a-time directly into the model's existing GPU parameters.
    Peak CPU RAM = size of the single largest tensor (~few hundred MB).
    """
    dtype_map = {
        "F16":  torch.float16,  "BF16": torch.bfloat16, "F32":  torch.float32,
        "I64":  torch.int64,    "I32":  torch.int32,     "I16":  torch.int16,
        "I8":   torch.int8,     "U8":   torch.uint8,     "BOOL": torch.bool,
    }
    matched = 0
    with open(filepath, "rb") as f:
        header_size = struct.unpack("<Q", f.read(8))[0]
        header      = json.loads(f.read(header_size))
        data_start  = 8 + header_size

        for key, meta in header.items():
            if key == "__metadata__" or key not in param_map:
                continue
            dt         = dtype_map[meta["dtype"]]
            shape      = meta["shape"]
            begin, end = meta["data_offsets"]

            f.seek(data_start + begin)
            raw    = f.read(end - begin)                           # read one tensor
            tensor = torch.frombuffer(bytearray(raw), dtype=dt).reshape(shape)
            param_map[key].data.copy_(tensor.to(dtype))            # copy to GPU in-place
            del raw, tensor                                        # free CPU immediately
            matched += 1
    return matched


print("Loading processor …")
processor = AutoProcessor.from_pretrained(MODEL_PATH)

# --- Initialise model fp16 DIRECTLY on GPU (no CPU RAM needed for weights) ---
print("Building model architecture directly on GPU …")
config = Blip2Config.from_pretrained(MODEL_PATH)

torch.set_default_dtype(torch.float16)      # all new tensors → fp16
with torch.device(device):                  # all new tensors → GPU
    model = Blip2ForConditionalGeneration(config)
torch.set_default_dtype(torch.float32)      # restore default immediately

# Build name → GPU parameter/buffer lookup
param_map = {n: p for n, p in model.named_parameters()}
param_map.update({n: b for n, b in model.named_buffers()})

# --- Stream real weights from disk, one tensor at a time → in-place GPU copy ---
shards = sorted(glob.glob(os.path.join(MODEL_PATH, "*.safetensors")))
for shard in shards:
    print(f"  Streaming {os.path.basename(shard)} …")
    matched = stream_shard_to_gpu(shard, param_map, device, dtype)
    print(f"    {matched} tensors loaded")

model.eval()
used  = torch.cuda.memory_allocated(0) / 1024**3
total = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"\nModel ready.  VRAM used: {used:.1f} / {total:.1f} GB")

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading processor …
Building model architecture directly on GPU …
  Streaming model-00001-of-00002.safetensors …
    995 tensors loaded
  Streaming model-00002-of-00002.safetensors …
    252 tensors loaded

Model ready.  VRAM used: 7.0 / 8.0 GB


## 4 — Zero-Shot Evaluation (Pretrained)

In [4]:
def to_device(inputs: dict) -> dict:
    """Move inputs to device; only cast floating-point tensors to dtype.
    Integer tensors (input_ids, attention_mask) must stay as Long."""
    result = {}
    for k, v in inputs.items():
        if v.is_floating_point():
            result[k] = v.to(device, dtype)
        else:
            result[k] = v.to(device)
    return result


def classify_response(response: str) -> int:
    lower = response.lower().strip()
    if any(kw in lower for kw in ("out-of-context", "no", "false")):
        return 1
    return 0


def run_eval(model, processor, samples, label="eval") -> dict:
    model.eval()
    results = []
    y_true, y_pred = [], []
    skipped = 0

    for i, item in enumerate(samples, start=1):
        try:
            image = Image.open(item["image_path"]).convert("RGB")
        except Exception as e:
            print(f"[{i}] SKIP — {e}")
            skipped += 1
            continue

        prompt = (
            f"Does this image match the caption: '{item['caption'][:100]}'? "
            "Answer:"
        )
        inputs = processor(images=image, text=prompt, return_tensors="pt")
        inputs = to_device(inputs)

        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS)

        response  = processor.tokenizer.decode(generated_ids[0], skip_special_tokens=True).strip()
        predicted = classify_response(response)

        y_true.append(item["label"])
        y_pred.append(predicted)
        results.append({
            "image_path": item["image_path"],
            "caption":    item["caption"],
            "label":      item["label"],
            "response":   response,
            "predicted":  predicted,
            "correct":    predicted == item["label"],
        })

        if i % 50 == 0 or i == len(samples):
            print(f"[{label}] {i}/{len(samples)}  running acc: {accuracy_score(y_true, y_pred):.3f}")

    metrics = {
        "accuracy":         accuracy_score(y_true, y_pred),
        "precision":        precision_score(y_true, y_pred, zero_division=0),
        "recall":           recall_score(y_true, y_pred, zero_division=0),
        "f1":               f1_score(y_true, y_pred, zero_division=0),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
        "evaluated":        len(y_true),
        "skipped":          skipped,
    }
    return {"metrics": metrics, "predictions": results}


print("Running zero-shot evaluation …")
pretrained_results = run_eval(model, processor, eval_samples, label="pretrained")
m = pretrained_results["metrics"]
print(f"\n── Pretrained results ──")
print(f"  Accuracy : {m['accuracy']:.4f}")
print(f"  Precision: {m['precision']:.4f}")
print(f"  Recall   : {m['recall']:.4f}")
print(f"  F1       : {m['f1']:.4f}")
print(f"  Skipped  : {m['skipped']}")
cm = m["confusion_matrix"]
print(f"  Confusion: TN={cm[0][0]} FP={cm[0][1]} FN={cm[1][0]} TP={cm[1][1]}")

Running zero-shot evaluation …
[pretrained] 50/500  running acc: 0.460
[pretrained] 100/500  running acc: 0.460
[pretrained] 150/500  running acc: 0.467
[pretrained] 200/500  running acc: 0.500
[pretrained] 250/500  running acc: 0.512
[pretrained] 300/500  running acc: 0.520
[pretrained] 350/500  running acc: 0.514
[pretrained] 400/500  running acc: 0.517
[pretrained] 450/500  running acc: 0.516
[pretrained] 500/500  running acc: 0.510

── Pretrained results ──
  Accuracy : 0.5100
  Precision: 0.5127
  Recall   : 0.4040
  F1       : 0.4519
  Skipped  : 0
  Confusion: TN=154 FP=96 FN=149 TP=101


## 5 — Fine-Tuning

### 5a — Freeze / Unfreeze Strategy

In [5]:
# Freeze everything
for p in model.parameters():
    p.requires_grad = False

# Unfreeze Q-Former + language_projection (LR_QFORMER)
for p in model.qformer.parameters():
    p.requires_grad = True
for p in model.language_projection.parameters():
    p.requires_grad = True

# Unfreeze last VISION_UNFREEZE vision encoder layers (LR_VISION)
encoder_layers = model.vision_model.encoder.layers
for layer in encoder_layers[-VISION_UNFREEZE:]:
    for p in layer.parameters():
        p.requires_grad = True

# Cast trainable params to fp32 — GradScaler cannot unscale fp16 gradients.
# Frozen params remain fp16 to save VRAM; autocast keeps the forward pass in fp16.
for p in model.parameters():
    if p.requires_grad:
        p.data = p.data.float()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")
used = torch.cuda.memory_allocated(0) / 1024**3
print(f"VRAM after unfreeze: {used:.1f} GB")

Trainable: 157,606,656 / 3,744,761,856 (4.21%)
VRAM after unfreeze: 7.3 GB


### 5b — Dataset & DataLoader

In [6]:
class PicsCanLieDataset(Dataset):
    def __init__(self, samples: list[dict]):
        self.items = samples

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        try:
            image = Image.open(item["image_path"]).convert("RGB")
        except Exception:
            image = Image.new("RGB", (224, 224), color=(128, 128, 128))
        prompt     = (
            f"Does this image match the caption: '{item['caption'][:100]}'? "
            "Answer:"
        )
        label_text = "out-of-context" if item["label"] == 1 else "real"
        return image, prompt, label_text


def collate_fn(batch):
    images, prompts, labels = zip(*batch)
    inputs = processor(
        images=list(images), text=list(prompts),
        return_tensors="pt", padding=True,
        truncation=True, max_length=MAX_INPUT_LEN,
    )
    label_enc = processor.tokenizer(
        list(labels), return_tensors="pt",
        padding="max_length", truncation=True, max_length=MAX_TARGET_LEN,
    )
    label_ids = label_enc["input_ids"].clone()
    label_ids[label_ids == processor.tokenizer.pad_token_id] = -100
    inputs["labels"] = label_ids
    return inputs


train_dataset = PicsCanLieDataset(train_samples)
loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=0, collate_fn=collate_fn,
    pin_memory=torch.cuda.is_available(),
)
print(f"Train samples: {len(train_dataset)}  |  Batches/epoch: {len(loader)}")

Train samples: 5000  |  Batches/epoch: 2500


### 5c — Training Loop

In [7]:
def vram_gb() -> float:
    return torch.cuda.memory_allocated(0) / 1024**3 if torch.cuda.is_available() else 0.0


qformer_params = [p for p in list(model.qformer.parameters()) +
                             list(model.language_projection.parameters())
                  if p.requires_grad]
vision_params  = [p for layer in encoder_layers[-VISION_UNFREEZE:]
                    for p in layer.parameters() if p.requires_grad]

optimizer = torch.optim.AdamW([
    {"params": qformer_params, "lr": LR_QFORMER},
    {"params": vision_params,  "lr": LR_VISION},
])

use_cuda = torch.cuda.is_available()
scaler   = torch.cuda.amp.GradScaler(enabled=use_cuda)

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()
    epoch_loss = 0.0
    t0 = time.time()

    for step, batch in enumerate(loader, start=1):
        batch = {
            k: v.to(device, dtype) if v.is_floating_point() else v.to(device)
            for k, v in batch.items()
        }

        with torch.cuda.amp.autocast(enabled=use_cuda):
            loss = model(**batch).loss / GRAD_ACCUM

        scaler.scale(loss).backward()

        if step % GRAD_ACCUM == 0 or step == len(loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        epoch_loss += loss.item() * GRAD_ACCUM

        if step % LOG_EVERY == 0:
            print(
                f"Epoch {epoch} | step {step}/{len(loader)} "
                f"| loss {epoch_loss / step:.4f} "
                f"| VRAM {vram_gb():.2f} GB "
                f"| {time.time() - t0:.0f}s"
            )

    avg = epoch_loss / len(loader)
    print(f"\nEpoch {epoch} complete — avg loss {avg:.4f}\n")

    ckpt = os.path.join(OUTPUT_DIR, f"epoch_{epoch}")
    os.makedirs(ckpt, exist_ok=True)
    model.save_pretrained(ckpt)
    processor.save_pretrained(ckpt)
    print(f"Checkpoint saved → {ckpt}\n")

print("Fine-tuning complete.")

C:\Users\Youssef Elghandour\AppData\Local\Temp\ipykernel_10980\3245479661.py:17: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler   = torch.cuda.amp.GradScaler(enabled=use_cuda)
C:\Users\Youssef Elghandour\AppData\Local\Temp\ipykernel_10980\3245479661.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_cuda):


Epoch 1 | step 50/2500 | loss 12.9275 | VRAM 9.08 GB | 159s
Epoch 1 | step 100/2500 | loss 12.6526 | VRAM 9.08 GB | 320s
Epoch 1 | step 150/2500 | loss 12.6447 | VRAM 9.08 GB | 483s
Epoch 1 | step 200/2500 | loss 12.5102 | VRAM 8.49 GB | 649s
Epoch 1 | step 250/2500 | loss 12.2185 | VRAM 9.08 GB | 813s
Epoch 1 | step 300/2500 | loss 11.8350 | VRAM 9.08 GB | 976s
Epoch 1 | step 350/2500 | loss 11.4429 | VRAM 9.08 GB | 1138s
Epoch 1 | step 400/2500 | loss 11.0116 | VRAM 8.49 GB | 1302s
Epoch 1 | step 450/2500 | loss 10.5471 | VRAM 9.08 GB | 1464s
Epoch 1 | step 500/2500 | loss 10.0501 | VRAM 9.08 GB | 1627s
Epoch 1 | step 550/2500 | loss 9.5236 | VRAM 9.08 GB | 1789s
Epoch 1 | step 600/2500 | loss 9.0173 | VRAM 8.49 GB | 1953s
Epoch 1 | step 650/2500 | loss 8.5594 | VRAM 9.08 GB | 2117s
Epoch 1 | step 700/2500 | loss 8.1215 | VRAM 9.08 GB | 2280s
Epoch 1 | step 750/2500 | loss 7.7286 | VRAM 9.08 GB | 2442s
Epoch 1 | step 800/2500 | loss 7.3653 | VRAM 8.49 GB | 2606s
Epoch 1 | step 850/25

Writing model shards:   0%|          | 0/1 [00:14<?, ?it/s]


MemoryError: 

## 6 — Post-Fine-Tuning Evaluation

In [ ]:
print("Running post-fine-tuning evaluation on the same eval split …")
finetuned_results = run_eval(model, processor, eval_samples, label="finetuned")
m = finetuned_results["metrics"]
print(f"\n── Fine-tuned results ──")
print(f"  Accuracy : {m['accuracy']:.4f}")
print(f"  Precision: {m['precision']:.4f}")
print(f"  Recall   : {m['recall']:.4f}")
print(f"  F1       : {m['f1']:.4f}")
print(f"  Skipped  : {m['skipped']}")
cm = m["confusion_matrix"]
print(f"  Confusion: TN={cm[0][0]} FP={cm[0][1]} FN={cm[1][0]} TP={cm[1][1]}")

## 7 — Compare & Save Results

In [ ]:
pre = pretrained_results["metrics"]
ft  = finetuned_results["metrics"]

print("=" * 50)
print(f"{'Metric':<12} {'Pretrained':>12} {'Fine-tuned':>12} {'Delta':>10}")
print("-" * 50)
for key in ("accuracy", "precision", "recall", "f1"):
    delta = ft[key] - pre[key]
    sign  = "+" if delta >= 0 else ""
    print(f"{key:<12} {pre[key]:>12.4f} {ft[key]:>12.4f} {sign}{delta:>9.4f}")
print("=" * 50)

output = {
    "config": {
        "model_path":    MODEL_PATH,
        "annotations":   ANNOTATIONS,
        "eval_samples":  EVAL_SAMPLES,
        "eval_seed":     EVAL_SEED,
        "train_samples": TRAIN_SAMPLES,
        "epochs":        EPOCHS,
        "batch_size":    BATCH_SIZE,
        "grad_accum":    GRAD_ACCUM,
    },
    "pretrained": pretrained_results,
    "finetuned":  finetuned_results,
}

with open(EVAL_OUTPUT, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f"\nResults saved → {EVAL_OUTPUT}")